In [ ]:
# Add uploaded Kaggle dataset code package to Python search path
import os
import sys

# Support both user-dataset and standard slug paths
possible_paths = [
    "/kaggle/input/datasets/someshchatterjee/gemma4-dev-agent-code",
    "/kaggle/input/gemma4-dev-agent-code",
]

DATASET_DIR = next((p for p in possible_paths if os.path.exists(p)), possible_paths[0])
if DATASET_DIR not in sys.path:
    sys.path.insert(0, DATASET_DIR)

print(f"Configured sys.path with DATASET_DIR: {DATASET_DIR}")

In [ ]:
from src.config.config_manager import ConfigManager
from src.data.chat_formatter import ChatFormatter
from src.data.dataset_builder import DatasetBuilder
from src.data.ingestor import DataIngestor
from src.data.patch_parser import PatchParser
from src.data.trajectory_augmentor import TrajectoryAugmentor
from src.data.trajectory_synthesiser import TrajectorySynthesiser
from src.data.trajectory_validator import TrajectoryValidator
from src.deployment.constraint_validator import ConstraintValidator
from src.deployment.submission_packager import SubmissionPackager
from src.evaluation.cv_evaluator import CVEvaluator  # noqa: F401
from src.evaluation.cv_splitter import CVSplitter
from src.evaluation.execution_budget import ExecutionBudget  # noqa: F401
from src.evaluation.metric_tracker import MetricTracker  # noqa: F401
from src.evaluation.mock_sandbox import MockSandbox  # noqa: F401
from src.evaluation.trajectory_executor import TrajectoryExecutor  # noqa: F401
from src.evaluation.verification_runner import VerificationRunner  # noqa: F401
from src.training.checkpoint_manager import CheckpointManager
from src.training.curriculum_scheduler import CurriculumScheduler
from src.training.reward_model import RewardModel  # noqa: F401
from src.training.rl_trainer import RLTrainerPipeline  # noqa: F401
from src.training.sft_trainer import SFTTrainerPipeline
from src.utils.telemetry_logger import TelemetryLogger
from src.utils.token_counter import TokenCounter

In [ ]:
config = ConfigManager.load(f"{DATASET_DIR}/configs/sft_config.yaml")
telemetry = TelemetryLogger(log_dir="/kaggle/working/logs", run_version="v0.1.0")

In [ ]:
# Wire up dependencies
token_counter = TokenCounter()
patch_parser = PatchParser()
ingestor = DataIngestor(config.get_data_paths())
synthesiser = TrajectorySynthesiser(
    ingestor=ingestor,
    patch_parser=patch_parser,
    token_counter=token_counter,
)
augmentor = TrajectoryAugmentor()
formatter = ChatFormatter(token_counter=token_counter)
validator = TrajectoryValidator(token_counter=token_counter)
splitter = CVSplitter()

builder = DatasetBuilder(
    ingestor=ingestor,
    synthesiser=synthesiser,
    augmentor=augmentor,
    formatter=formatter,
    validator=validator,
    splitter=splitter,
)

dataset = builder.build()
print(f"Dataset built: {len(dataset)} examples")
print(f"Columns: {dataset.column_names}")

In [ ]:
checkpoint_mgr = CheckpointManager()
curriculum = CurriculumScheduler()

sft_pipeline = SFTTrainerPipeline(
    checkpoint_mgr=checkpoint_mgr,
    telemetry=telemetry,
    curriculum=curriculum,
)

sft_config = config.get_sft_config()
sft_adapter_path = sft_pipeline.run(sft_config, dataset)
print(f"SFT adapter saved to: {sft_adapter_path}")

In [ ]:
# Uncomment when ready for RL training:
# rl_config = config.get_rl_config()
# reward_model = RewardModel(telemetry=telemetry)
# rl_pipeline = RLTrainerPipeline(
#     reward_model=reward_model,
#     checkpoint_mgr=checkpoint_mgr,
#     telemetry=telemetry,
# )
# tasks = ingestor.load_tasks()
# rl_adapter_path = rl_pipeline.run(rl_config, tasks)
# print(f"RL adapter saved to: {rl_adapter_path}")

In [ ]:
# Uncomment to run quick CV:
# sandbox = MockSandbox()
# budget = ExecutionBudget()
# executor = TrajectoryExecutor(budget=budget, telemetry=telemetry)
# verifier = VerificationRunner(sandbox=sandbox, telemetry=telemetry)
# tracker = MetricTracker()
#
# evaluator = CVEvaluator(
#     splitter=splitter,
#     executor=executor,
#     verifier=verifier,
#     tracker=tracker,
#     telemetry=telemetry,
# )
# tasks = ingestor.load_tasks()
# report = evaluator.evaluate_all_folds(sft_adapter_path, tasks, version="v0.1.0")
# print(f"CV Resolution Rate: {report.aggregate_resolution_rate:.1%}")

In [ ]:
import os

from src.config.deploy_config import DeployConfig

deploy_config = DeployConfig(
    templates_dir=f"{DATASET_DIR}/submission_templates",
    output_dir="/kaggle/working/submission",
    zip_path="/kaggle/working/submission.zip",
    adapters=[{
        "name": "coder_lora",
        "checkpoint_path": str(sft_adapter_path),
    }],
)

validator = ConstraintValidator()
packager = SubmissionPackager(validator=validator)
zip_path = packager.package(deploy_config)
print(f"Submission packaged: {zip_path}")

size_mb = os.path.getsize(zip_path) / 1e6
print(f"Submission size: {size_mb:.1f} MB")